# Build Classified ChatDev Summarized Dataset

Convert playbooks under `data/classified_chatdev_playbook` into summarized JSON under `data/classified_chatdev_summarized`, preserving the label-directory layout. Duplicate samples are detected by file name, so the same playbook name is summarized once and copied to the other matching label directories. Repeated prompt/output texts are also cached so they do not call the LLM twice.

In [1]:
from pathlib import Path
import copy
import hashlib
import json
import shutil
import sys
import time

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_ROOT = PROJECT_ROOT / "data" / "classified_chatdev_playbook"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "classified_chatdev_summarized"
DATASET_JSON = PROJECT_ROOT / "chatdev_dataset.json"
TRAJECTORY_DIR = INPUT_ROOT / "trajectory"
CACHE_PATH = OUTPUT_ROOT / "_llm_summarizer_cache.json"
MANIFEST_PATH = OUTPUT_ROOT / "_manifest.json"

# Tune these before running if needed.
MODEL = "gpt-4o"
MAX_RETRIES = 3
OVERWRITE = False
CONTINUE_ON_ERROR = True
MAX_FILES = None  # set to an integer for a small test run

INPUT_ROOT, OUTPUT_ROOT

(WindowsPath('d:/Works/code/winter-like-ai/ChatDev/data/classified_chatdev_playbook'),
 WindowsPath('d:/Works/code/winter-like-ai/ChatDev/data/classified_chatdev_summarized'))

In [2]:
from chatdev.analyzer.llm_summarizer import LLMSummarizer
from chatdev.analyzer.logprob_consistency import chatdev_filename_sort_key, infer_user_task_for_path, load_user_task_map


def text_sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def summarized_output_path(src_path: Path) -> Path:
    rel = src_path.relative_to(INPUT_ROOT)
    name = rel.name
    if name.endswith("_playbook.json"):
        name = name[:-len("_playbook.json")] + "_summarized.json"
    else:
        name = rel.stem + "_summarized.json"
    return OUTPUT_ROOT / rel.parent / name


def load_json(path: Path, default):
    if not path.exists():
        return copy.deepcopy(default)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def inject_user_demand(summary_path: Path, source_path: Path, user_task_map: dict[str, str]) -> None:
    user_demand = infer_user_task_for_path(str(source_path), user_task_map=user_task_map)
    if not user_demand or not summary_path.exists():
        return
    payload = load_json(summary_path, {})
    changed = False
    for interactions in payload.values():
        if not isinstance(interactions, list):
            continue
        for entry in interactions:
            if isinstance(entry, dict) and entry.get("user_demand") != user_demand:
                entry.pop("user_task", None)
                entry["user_demand"] = user_demand
                changed = True
    if changed:
        write_json(summary_path, payload)


class CachedLLMSummarizer(LLMSummarizer):
    """LLMSummarizer with persistent prompt/output caches keyed by text hash."""

    def __init__(self, cache_path: Path, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.cache_path = cache_path
        self.cache = load_json(cache_path, {"prompt": {}, "output": {}})
        self.cache_stats = {"prompt_hits": 0, "prompt_misses": 0, "output_hits": 0, "output_misses": 0}

    def summarize_prompt(self, prompt_text: str):
        key = text_sha256(prompt_text or "")
        cached = self.cache["prompt"].get(key)
        if cached is not None:
            self.cache_stats["prompt_hits"] += 1
            return copy.deepcopy(cached)
        self.cache_stats["prompt_misses"] += 1
        result = super().summarize_prompt(prompt_text)
        self.cache["prompt"][key] = copy.deepcopy(result)
        return result

    def summarize_output(self, output_text: str):
        key = text_sha256(output_text or "")
        cached = self.cache["output"].get(key)
        if cached is not None:
            self.cache_stats["output_hits"] += 1
            return copy.deepcopy(cached)
        self.cache_stats["output_misses"] += 1
        result = super().summarize_output(output_text)
        self.cache["output"][key] = copy.deepcopy(result)
        return result

    def save_cache(self) -> None:
        write_json(self.cache_path, self.cache)


def discover_playbook_paths() -> list[Path]:
    paths = sorted(INPUT_ROOT.rglob("*.json"))
    paths = [p for p in paths if p.is_file()]
    if MAX_FILES is not None:
        paths = paths[:MAX_FILES]
    return paths


def group_playbooks_by_filename(paths: list[Path]) -> dict[str, list[Path]]:
    groups = {}
    for path in paths:
        groups.setdefault(path.name, []).append(path)
    return {name: sorted(items, key=canonical_sort_key) for name, items in sorted(groups.items(), key=lambda item: chatdev_filename_sort_key(item[0]))}


def canonical_sort_key(path: Path):
    # Prefer trajectory as the canonical copy when present because it contains
    # the full 130-ish sample set; label folders then receive copied outputs.
    rel = path.relative_to(INPUT_ROOT)
    return (chatdev_filename_sort_key(rel.name), 0 if rel.parts[0] == "trajectory" else 1, str(rel))


print(f"Input exists: {INPUT_ROOT.exists()} -> {INPUT_ROOT}")
all_paths = discover_playbook_paths()
filename_groups = group_playbooks_by_filename(all_paths)
user_task_map = load_user_task_map(dataset_path=str(DATASET_JSON), trajectory_dir=str(TRAJECTORY_DIR))
print(f"Found playbook paths: {len(all_paths)}")
print(f"Unique playbook filenames: {len(filename_groups)}")
print(f"User demand mappings: {len(user_task_map)}")

Input exists: True -> d:\Works\code\winter-like-ai\ChatDev\data\classified_chatdev_playbook
Found playbook paths: 448
Unique playbook filenames: 130


In [3]:
# Build an initial filename map from already completed outputs.
# This lets a resumed run copy same-name playbooks without API calls.
filename_to_summary = {}
existing_pairs = 0

for src_path in discover_playbook_paths():
    dst_path = summarized_output_path(src_path)
    if dst_path.exists():
        filename_to_summary.setdefault(src_path.name, dst_path)
        existing_pairs += 1

print(f"Existing summarized outputs found: {existing_pairs}")
print(f"Unique filenames already summarized: {len(filename_to_summary)}")

Existing summarized outputs found: 12
Unique filenames already summarized: 4


In [4]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summarizer = CachedLLMSummarizer(
    cache_path=CACHE_PATH,
    model=MODEL,
    max_retries=MAX_RETRIES,
)

manifest = {
    "input_root": str(INPUT_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "model": MODEL,
    "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "overwrite": OVERWRITE,
    "records": [],
}

all_paths = discover_playbook_paths()
filename_groups = group_playbooks_by_filename(all_paths)
print(f"Processing {len(filename_groups)} unique playbook filenames across {len(all_paths)} paths")

for index, (filename, same_name_paths) in enumerate(filename_groups.items(), start=1):
    canonical_src = same_name_paths[0]
    canonical_dst = summarized_output_path(canonical_src)
    canonical_rel = canonical_src.relative_to(INPUT_ROOT)
    record = {
        "index": index,
        "filename": filename,
        "canonical_source": str(canonical_rel),
        "canonical_output": str(canonical_dst.relative_to(OUTPUT_ROOT)),
        "path_count": len(same_name_paths),
        "paths": [str(path.relative_to(INPUT_ROOT)) for path in same_name_paths],
    }

    try:
        if canonical_dst.exists() and not OVERWRITE:
            record["status"] = "skipped_existing"
            filename_to_summary.setdefault(filename, canonical_dst)
            inject_user_demand(canonical_dst, canonical_src, user_task_map)
            print(f"[{index}/{len(filename_groups)}] SKIP existing {canonical_rel}")
        elif filename in filename_to_summary and filename_to_summary[filename].exists():
            canonical_dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(filename_to_summary[filename], canonical_dst)
            inject_user_demand(canonical_dst, canonical_src, user_task_map)
            record["status"] = "copied_filename_duplicate"
            record["duplicate_of"] = str(filename_to_summary[filename].relative_to(OUTPUT_ROOT))
            print(f"[{index}/{len(filename_groups)}] COPY filename duplicate {canonical_rel}")
        else:
            print(f"[{index}/{len(filename_groups)}] SUMMARIZE {canonical_rel}")
            summarizer.summarize_playbook(str(canonical_src), output_path=str(canonical_dst), verbose=False)
            inject_user_demand(canonical_dst, canonical_src, user_task_map)
            summarizer.save_cache()
            filename_to_summary[filename] = canonical_dst
            record["status"] = "summarized"

        copied_outputs = []
        source_summary = filename_to_summary.get(filename, canonical_dst)
        for duplicate_src in same_name_paths:
            duplicate_dst = summarized_output_path(duplicate_src)
            if duplicate_dst == source_summary:
                continue
            if duplicate_dst.exists() and not OVERWRITE:
                continue
            duplicate_dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_summary, duplicate_dst)
            inject_user_demand(duplicate_dst, duplicate_src, user_task_map)
            copied_outputs.append(str(duplicate_dst.relative_to(OUTPUT_ROOT)))
        record["copied_outputs"] = copied_outputs
    except Exception as exc:
        record["status"] = "error"
        record["error"] = repr(exc)
        summarizer.save_cache()
        print(f"[{index}/{len(filename_groups)}] ERROR {canonical_rel}: {exc}")
        if not CONTINUE_ON_ERROR:
            manifest["records"].append(record)
            write_json(MANIFEST_PATH, manifest)
            raise

    manifest["records"].append(record)
    if index % 5 == 0:
        summarizer.save_cache()
        write_json(MANIFEST_PATH, manifest)

summarizer.save_cache()
manifest["finished_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
manifest["summarizer_stats"] = summarizer.stats
manifest["cache_stats"] = summarizer.cache_stats
write_json(MANIFEST_PATH, manifest)

print("Done.")
print(json.dumps({
    "records": len(manifest["records"]),
    "statuses": {s: sum(1 for r in manifest["records"] if r.get("status") == s) for s in sorted({r.get("status") for r in manifest["records"]})},
    "summarizer_stats": summarizer.stats,
    "cache_stats": summarizer.cache_stats,
    "manifest": str(MANIFEST_PATH),
}, ensure_ascii=False, indent=2))

Processing 130 unique playbook filenames across 448 paths
[1/130] SKIP existing trajectory\ChatDev_ProgramDev2_GPT4o_0_playbook.json
[2/130] SKIP existing trajectory\ChatDev_ProgramDev2_GPT4o_10_playbook.json
[3/130] SKIP existing trajectory\ChatDev_ProgramDev2_GPT4o_11_playbook.json
[4/130] SKIP existing trajectory\ChatDev_ProgramDev2_GPT4o_12_playbook.json
[5/130] SUMMARIZE trajectory\ChatDev_ProgramDev2_GPT4o_13_playbook.json
[6/130] SUMMARIZE trajectory\ChatDev_ProgramDev2_GPT4o_14_playbook.json
[7/130] SUMMARIZE trajectory\ChatDev_ProgramDev2_GPT4o_15_playbook.json
[8/130] SUMMARIZE trajectory\ChatDev_ProgramDev2_GPT4o_16_playbook.json
[9/130] SUMMARIZE trajectory\ChatDev_ProgramDev2_GPT4o_17_playbook.json
[10/130] SUMMARIZE trajectory\ChatDev_ProgramDev2_GPT4o_18_playbook.json
[11/130] SUMMARIZE trajectory\ChatDev_ProgramDev2_GPT4o_19_playbook.json
[12/130] SUMMARIZE trajectory\ChatDev_ProgramDev2_GPT4o_1_playbook.json
[13/130] SUMMARIZE trajectory\ChatDev_ProgramDev2_GPT4o_20_pl

In [5]:
# Quick structural check: every input playbook should have one output summarized JSON.
missing = []
for src_path in discover_playbook_paths():
    dst_path = summarized_output_path(src_path)
    if not dst_path.exists():
        missing.append(str(src_path.relative_to(INPUT_ROOT)))

print(f"Missing outputs: {len(missing)}")
if missing[:20]:
    print(json.dumps(missing[:20], ensure_ascii=False, indent=2))

# Show one sample output path and its top-level roles.
sample_outputs = sorted(OUTPUT_ROOT.rglob("*_summarized.json"))
if sample_outputs:
    sample = sample_outputs[0]
    with sample.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    print(f"Sample: {sample.relative_to(OUTPUT_ROOT)}")
    print(list(payload)[:10])

Missing outputs: 0
Sample: 0.0\ChatDev_ProgramDev2_GPT4o_0_summarized.json
['Chief Product Officer', 'Chief Technology Officer', 'Programmer', 'Code Reviewer', 'Chief Executive Officer']
